# LBG REASON — Ollama éphémère (Colab T4 / Kaggle)

**PoC** pour brancher un GPU cloud gratuit sur Cortex via `LBG_REASON_EPHEMERAL_*`.

- Doc : `packages/cortex/docs/reason_ephemeral_colab.md`
- Comparatif free : `packages/cortex/docs/llm_cloud_free_vs_local.md`
- Secours API (sans notebook) : `packages/cortex/docs/reason_cloud_free_fallback.md`
- **Pas** pour la prod H24 (`.140` / Team). Préférer Groq (`LBG_REASON_CLOUD_*`) si besoin stable.
- Runtime : **Colab GPU → T4** (ou Kaggle GPU P100/T4 — mêmes cellules).
- Modèle défaut T4 : `huihui_ai/qwen3.5-abliterated:9b` (override `LBG_COLAB_MODEL`).

À la fin, le notebook affiche un bloc `export …` à coller en local.


In [ ]:
# 0) GPU check (Colab / Kaggle / Lightning)
# nvidia-smi -L → « GPU 0: Tesla T4 (...) » sans le mot « NVIDIA ».
import re, shutil, subprocess

out = subprocess.getoutput("nvidia-smi -L")
print(out or "PAS DE GPU")
low = (out or "").lower()
ok = bool(
    shutil.which("nvidia-smi")
    and out
    and "failed" not in low
    and "not found" not in low
    and re.search(r"\b(tesla|t4|p100|a100|l4|v100|rtx|geforce|quadro|nvidia)\b", low)
)
assert ok, "Pas de GPU. Colab: Runtime→T4 | Kaggle: Accelerator→GPU | Lightning: Studio GPU"
print("OK GPU")


In [ ]:
import os, subprocess, time, pathlib, shutil
# 1) Install Ollama + cloudflared

def sh(cmd, check=True):
    print("+", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-2000:])
    if r.stderr:
        print(r.stderr[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"cmd failed ({r.returncode}): {cmd}")
    return r

if not pathlib.Path("/usr/local/bin/ollama").exists() and not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh")
else:
    print("ollama déjà présent")

# cloudflared (tunnel rapide, sans compte)
cf = pathlib.Path("/usr/local/bin/cloudflared")
if not cf.exists():
    sh("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    sh("chmod +x /usr/local/bin/cloudflared")
print("ok:", shutil.which("ollama"), shutil.which("cloudflared") or str(cf))


In [ ]:
# 2) Démarrer Ollama + pull modèle T4-friendly
import os, subprocess, time, secrets, shutil

def _resolve_ollama() -> str:
    """Chemin réel d'ollama (Colab/Kaggle : pas toujours /usr/local/bin)."""
    import pathlib, shutil, subprocess
    for cand in (
        shutil.which("ollama"),
        "/usr/local/bin/ollama",
        "/usr/bin/ollama",
        str(pathlib.Path.home() / "bin" / "ollama"),
    ):
        if cand and pathlib.Path(cand).is_file():
            return cand
    # Dernier recours : which via shell (PATH notebook ≠ PATH install)
    wh = subprocess.getoutput("command -v ollama || which ollama || true").strip().splitlines()
    for line in wh:
        p = line.strip()
        if p and pathlib.Path(p).is_file():
            return p
    raise FileNotFoundError(
        "ollama introuvable — relancer la cellule d'install (curl install.sh), "
        "puis vérifier: !command -v ollama"
    )


MODEL = os.environ.get("LBG_COLAB_MODEL", "huihui_ai/qwen3.5-abliterated:9b")  # alt: gemma-4-abliterated:12b | qwen3.5-abliterated:2B | legacy 8b-v2
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# kill éventuel
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)
ollama_bin = _resolve_ollama()
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
for _ in range(30):
    r = subprocess.run(["curl", "-fsS", "http://127.0.0.1:11434/api/tags"], capture_output=True)
    if r.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama ne répond pas sur :11434")

print("Pull", MODEL, "…")
sh(f"{ollama_bin} pull {MODEL}")
print(subprocess.getoutput(f"{ollama_bin} list"))


In [ ]:
# 3) Proxy Bearer → Ollama OpenAI-compat (Ollama n'a pas d'auth native)
import json, secrets, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

LBG_TOKEN = secrets.token_urlsafe(24)
PROXY_PORT = 8088
OLLAMA = "http://127.0.0.1:11434"

class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        print("[proxy]", fmt % args)

    def _unauthorized(self):
        self.send_response(401)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(b'{"error":"unauthorized"}')

    def _check_auth(self):
        auth = self.headers.get("Authorization", "")
        return auth == f"Bearer {LBG_TOKEN}"

    def _proxy(self, method: str):
        if not self._check_auth():
            return self._unauthorized()
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length) if length else None
        target = OLLAMA + self.path
        req = Request(target, data=body, method=method)
        req.add_header("Content-Type", self.headers.get("Content-Type", "application/json"))
        try:
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                self.send_response(resp.status)
                self.send_header("Content-Type", resp.headers.get("Content-Type", "application/json"))
                self.send_header("Content-Length", str(len(data)))
                self.end_headers()
                self.wfile.write(data)
        except HTTPError as e:
            data = e.read()
            self.send_response(e.code)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(data)
        except URLError as e:
            self.send_response(502)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"error": str(e)}).encode())

    def do_GET(self):
        self._proxy("GET")

    def do_POST(self):
        self._proxy("POST")

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "Authorization, Content-Type")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.end_headers()

httpd = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), Handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"Proxy auth sur http://127.0.0.1:{PROXY_PORT} → Ollama")
print("TOKEN (copié dans le bloc env plus bas):", LBG_TOKEN)


In [ ]:
# 4) Tunnel Cloudflare quick (URL publique éphémère)
import re, subprocess, time, shutil

cf = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
# stop previous
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)

proc_cf = subprocess.Popen(
    [cf, "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

PUBLIC_URL = None
deadline = time.time() + 60
buf = []
while time.time() < deadline and PUBLIC_URL is None:
    line = proc_cf.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    buf.append(line)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0).rstrip("/")
        break

if not PUBLIC_URL:
    print("".join(buf[-40:]))
    raise RuntimeError("Tunnel Cloudflare : URL non trouvée (réessayer la cellule)")

print("PUBLIC_URL =", PUBLIC_URL)


In [ ]:
# 5) Smoke + bloc env pour Cortex
import subprocess, json, textwrap

# smoke /v1/models
smoke = subprocess.run(
    [
        "curl", "-fsS",
        "-H", f"Authorization: Bearer {LBG_TOKEN}",
        f"{PUBLIC_URL}/v1/models",
    ],
    capture_output=True, text=True,
)
print("HTTP", smoke.returncode)
print((smoke.stdout or smoke.stderr)[:800])

block = textwrap.dedent(f'''
# --- LBG REASON ephemeral (session Colab — ne pas push sur .140) ---
export LBG_REASON_EPHEMERAL_ENABLED=1
export LBG_REASON_EPHEMERAL_BASE_URL="{PUBLIC_URL}"
export LBG_REASON_EPHEMERAL_API_KEY="{LBG_TOKEN}"
export LBG_REASON_EPHEMERAL_MODEL="{MODEL}"
export LBG_REASON_EPHEMERAL_TIMEOUT_S=45
export LBG_REASON_EPHEMERAL_BEFORE_CLOUD=1
# Laissez tourner cette cellule / runtime tant que vous testez.
''').strip()
print("\n" + "=" * 60)
print(block)
print("=" * 60)
print("\nSonde locale (WSL) :")
print(f'curl -sS -H "Authorization: Bearer $LBG_REASON_EPHEMERAL_API_KEY" \\')
print(f'  "$LBG_REASON_EPHEMERAL_BASE_URL/v1/models" | head')


## Arrêt

Runtime → **Disconnect and delete runtime** quand le PoC est fini.  
L’URL `trycloudflare.com` et le token deviennent invalides — c’est voulu.

Sans GPU local durable : utilisez `LBG_REASON_CLOUD_*` (Groq), pas ce notebook en permanence.
